# Stage 04 — DoubleML Extension

Estimate causal effect with Double/Debiased Machine Learning.
Follow `skills/stage_04.md` for detailed instructions.

In [ ]:
from paths import *
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import doubleml as dml
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV

df   = pd.read_parquet(str(DATASET_PATH))
spec = json.loads(PAPER_SPEC.read_text())
rep  = json.loads(REPLICATION_RESULTS.read_text())

outcome   = spec['identification']['outcome_variable']
treatment = spec['identification']['treatment_variable']
instrument = spec['identification'].get('instrument')
controls  = spec['identification']['controls']
id_type   = spec['identification']['type']
model_type = spec['dml']['model_type']  # PLIV, PLR, or IRM

print(f'DML model: {model_type}  |  N obs: {df[controls + [outcome, treatment]].dropna().shape[0]}')

In [ ]:
# --- AGENT: prepare data for DoubleML ---
# Drop rows with missing values in key variables
# Build DoubleMLData object based on model_type
# See skills/stage_04.md for exact code pattern

key_cols = [outcome, treatment] + controls + ([instrument] if instrument else [])
df_clean = df[key_cols].dropna().reset_index(drop=True)
print(f'Clean dataset: {df_clean.shape}')

# dml_data = dml.DoubleMLData(df_clean, y_col=outcome, d_col=treatment,
#                              z_col=instrument, x_cols=controls)
pass

In [ ]:
# --- AGENT: fit DML with RandomForest learner ---
# Use n_folds=5, n_rep=3
# Store coef, se, ci, pval, nuisance r2
rf_results = {}

In [ ]:
# --- AGENT: fit DML with LassoCV learner ---
lasso_results = {}

In [ ]:
# --- Write dml_results.json ---
# Determine preferred learner (RF unless RF r2 < 0.1)
preferred = 'RandomForest'  # agent updates if needed

dml_results = {
    'model_type': model_type,
    'n_obs': int(df_clean.shape[0]),
    'n_folds': 5,
    'n_rep': 3,
    'learners': {
        'RandomForest': rf_results,
        'LassoCV': lasso_results
    },
    'preferred_learner': preferred,
    'preferred_coef':   rf_results.get('coef'),
    'preferred_ci_lo':  rf_results.get('ci_lo'),
    'preferred_ci_hi':  rf_results.get('ci_hi'),
    'sign_change': False  # agent sets True if sign differs from published
}
DML_RESULTS.write_text(json.dumps(dml_results, indent=2))
print(f'✓ {DML_RESULTS}')

In [ ]:
# --- AGENT: forest plot ---
# Coefficient comparison: OLS baseline, IV baseline, DML (RF), DML (Lasso)
# Save to FIGURES_DIR / 'forest_plot.pdf' and 'forest_plot.png'
# See skills/stage_04.md for plot spec
pass

In [ ]:
# --- AGENT: LaTeX comparison table ---
# table_dml.tex: OLS | IV | DML(RF) | DML(Lasso)
# Save to TABLES_DIR / 'table_dml.tex'
pass

In [ ]:
# --- AGENT: GATE — Group Average Treatment Effects ---
# See skills/stage_04.md section 6 and skills/references/doubleml/heterogeneity.rst
#
# 1. Choose grouping_var = controls[0] if continuous, else treatment_variable
# 2. Create 4 quartile groups using pd.qcut
# 3. Call obj_preferred.gate(groups=groups_df)
# 4. Call gate.confint(level=0.95, joint=True)  ← jointly valid CIs via gaussian multiplier bootstrap
# 5. Write hte_results.json with schema from skills/stage_04.md
#    - heterogeneity_detected = True if any two GATE CIs do not overlap
#
# Works for PLR, PLIV, and IRM model types.
hte_results = {}  # agent fills this
HTE_RESULTS = RESULTS_DIR / 'hte_results.json'
# HTE_RESULTS.write_text(json.dumps(hte_results, indent=2))
pass

In [ ]:
# --- AGENT: CATE — Conditional Average Treatment Effects (PLR/PLIV only) ---
# Skip this cell for IRM models — set cate_summary = null in hte_results.json instead.
# See skills/references/doubleml/heterogeneity.rst for full API details.
#
# 1. Build B-spline basis: patsy.dmatrix(f"bs({grouping_var}, df=5) - 1", df_clean)
# 2. Call obj_preferred.cate(basis=basis_df)
# 3. Call cate.confint(level=0.95, joint=True)
# 4. Compute summary stats (mean, sd, min, max of point estimates)
# 5. Add cate_summary to hte_results.json (update the file written in previous cell)
pass

In [ ]:
# --- AGENT: GATE plot + LaTeX table ---
# See skills/stage_04.md sections 6d and 6e
#
# Plot: horizontal coefficient plot, groups on y-axis, coef ± joint CI on x-axis
#   - Same style as forest_plot (grey/blue colour scheme, dashed line at 0)
#   - Save as FIGURES_DIR / 'gate_plot.pdf' and 'gate_plot.png'
#
# Table: LaTeX table with columns: Group | Coef. | 95% CI (joint)
#   - Caption: "Group Average Treatment Effects (GATE)"
#   - Save as TABLES_DIR / 'table_gate.tex'
pass